# T2 — Semantic ICD linking (bge-m3 + reranker) trên Colab

Build embedding index cho ICD-10-CM rồi đo hit@k trên dev gold.

**Bước 1:** Runtime → Change runtime type → **T4 GPU** → Save.
**Bước 2:** Runtime → **Run all** (mỗi cell tự clone repo nếu thiếu — không cần token, repo public).

Xong gửi lại khối `=== ICD-10 (CHẨN_ĐOÁN) ===` cho P2.

In [11]:
# 1) Kiểm tra GPU (phải thấy Tesla T4). Nếu lỗi -> chưa bật GPU ở Runtime.
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-d3c462d6-f8e3-d2cd-598e-f713e50656d2)


In [12]:
# 2) Clone repo (PUBLIC, không cần token) nhánh Duckun + cài FlagEmbedding.
import os
if not os.path.isdir('/content/viettel-medreason'):
    !git clone -b Duckun https://github.com/tienndat1904/viettel-medreason.git /content/viettel-medreason
%cd /content/viettel-medreason
!git pull
!git log --oneline -1
!pip -q install FlagEmbedding==1.3.2

Cloning into '/content/viettel-medreason'...
remote: Enumerating objects: 494, done.
remote: Counting objects: 100% (494/494), done.
remote: Compressing objects: 100% (325/325), done.
remote: Total 494 (delta 191), reused 438 (delta 150), pack-reused 0 (from 0)
Receiving objects: 100% (494/494), 3.82 MiB | 20.68 MiB/s, done.
Resolving deltas: 100% (191/191), done.
/content/viettel-medreason
Already up to date.
27bd2e5 (HEAD -> Duckun, origin/Duckun) Merge remote-tracking branch 'origin/Duckun' into Duckun


In [13]:
# 3) Build index (lần đầu tải bge-m3 ~2.3GB, ~vài phút trên T4). Hết VRAM -> thêm --batch-size 64
import os
if not os.path.isdir('/content/viettel-medreason'):
    !git clone -b Duckun https://github.com/tienndat1904/viettel-medreason.git /content/viettel-medreason
%cd /content/viettel-medreason
!python src/kb/build_icd_index.py

/content/viettel-medreason
2026-07-02 09:41:46.016217: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 100% 444/444 [00:00<00:00, 3.48MB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 6.67MB/s]
tokenizer.json: 100% 17.1M/17.1M [00:00<00:00, 36.7MB/s]
special_tokens_map.json: 100% 964/964 [00:00<00:00, 9.67MB/s]
Fetching 30 files:   0% 0/30 [00:00<?, ?it/s]
README.md: 0.00B [00:00, ?B/s]

config.json: 100% 687/687 [00:00<00:00, 583kB/s]


README.md: 15.8kB [00:00, 2.32MB/s]



.DS_Store:   0% 0.00/6.15k [00:00<?, ?B/s]
.gitattributes: 1.63kB [00:00, 1.39MB/s]
.DS_Store: 100% 6.15k/6.15k [00:00<00:00, 5.61MB/s]


config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 566kB/s]


config.json: 100% 191/191 [00

In [14]:
# 4) Bật semantic + đo trên 30 file dev gold (embed query + rerank cần GPU).
import os
if not os.path.isdir('/content/viettel-medreason'):
    !git clone -b Duckun https://github.com/tienndat1904/viettel-medreason.git /content/viettel-medreason
%cd /content/viettel-medreason
!sed -i 's/backend: lexical/backend: semantic/' configs/config.yaml
!python src/eval/eval_linking.py

/content/viettel-medreason
[linker] backend=semantic icd_mode=semantic | icd=kb(98186) rxnorm=kb(47719) | synonyms=53 brands=4150
2026-07-02 09:45:23.369685: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Fetching 30 files: 100% 30/30 [00:00<00:00, 19778.23it/s]
tokenizer_config.json: 1.17kB [00:00, 6.61MB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 7.63MB/s]
tokenizer.json: 100% 17.1M/17.1M [00:00<00:00, 93.3MB/s]
special_tokens_map.json: 100% 964/964 [00:00<00:00, 7.63MB/s]
config.json: 100% 795/795 [00:00<00:00, 6.27MB/s]
model.safetensors: 100% 2.27G/2.27G [00:18<00:00, 121MB/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to

In [15]:
# 5) (Tuỳ chọn) Lưu index vào Google Drive để phiên sau khỏi build lại.
%cd /content/viettel-medreason
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/viettel_kb
!cp data/kb/icd10cm_emb.npy data/kb/icd10cm_index_meta.parquet /content/drive/MyDrive/viettel_kb/
print('Đã lưu index vào Drive/MyDrive/viettel_kb')

/content/viettel-medreason
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã lưu index vào Drive/MyDrive/viettel_kb


## Gửi lại cho P2
Copy khối kết quả ở cell 4:
```
=== ICD-10 (CHẨN_ĐOÁN) ===
  hit@k ... = 0.???
  top1  ... = 0.???
```
So với bản lexical hiện tại **0.495** để quyết định tinh chỉnh `icd_top_k_retrieve` / `icd_top_k_return` / `min_confidence`.